In [ ]:
import fitz
import os
import shutil
from docx import Document
import re
import cv2
import json
import csv
from functools import cmp_to_key
import math

In [4]:
pdf_dir='./data/Documents/'
transcripts_dir='./data/Transcriptions/'
pages_dir='./intermediates/raw pages/'
txt_dir='./intermediates/raw txt/'

In [11]:
def extract_pages(docx_path):

    doc = Document(docx_path)

    pages = {}
    current_page = None
    buffer = []
    page_pattern = re.compile(r"\bPDF p(\d+)(?:\s*[-–—]\s*(left|right))?\b")
    for para in doc.paragraphs:
        text = para.text.strip()

        match = page_pattern.match(text)

        if match:
            if current_page:
                pages[current_page] = "\n".join(buffer)
                buffer = []

            current_page = f"page_{int(match.group(1))}"+(f"_{match.group(2)}" if match.group(2) else "")
            continue

        if current_page:
            buffer.append(text)

    if current_page:
        pages[current_page] = "\n".join(buffer)

    return pages
a=extract_pages(f'{transcripts_dir}PORCONES.23.5 - 1628 transcription.docx')
for k,v in a.items():
    print(f"{k}: {len(v)} characters")

page_1: 486 characters
page_2_left: 1620 characters
page_3_right: 1953 characters
page_3_left: 1981 characters
page_4_left: 1939 characters


In [14]:
for filename in os.listdir(transcripts_dir):
    if filename.endswith('.docx'):
        print(f"Processing {filename}...")
        pages = extract_pages(os.path.join(transcripts_dir, filename))
        txt_subdir=os.path.join(txt_dir, os.path.splitext(filename)[0])
        os.makedirs(txt_subdir, exist_ok=True)
        for page, content in pages.items():
            with open(os.path.join(txt_subdir, f"{page}.txt"), 'w', encoding='utf-8') as f:
                f.write(content)

Processing Buendia - Instruccion transcription.docx...
Processing Covarrubias - Tesoro lengua transcription.docx...
Processing Guardiola - Tratado nobleza transcription.docx...
Processing PORCONES.228.38 - 1646 transcription.docx...
Processing PORCONES.23.5 - 1628 transcription.docx...
Processing PORCONES.748.6 – 1650 Transcription.docx...


In [ ]:
for filename in os.listdir(pdf_dir):
    if filename.endswith('.pdf'):
        pdf_path = os.path.join(pdf_dir, filename)
        doc = fitz.open(pdf_path)
        base_name = os.path.splitext(filename)[0]
        pages_subdir = os.path.join(pages_dir, base_name)
        os.makedirs(pages_subdir, exist_ok=True)
        for page_num in range(doc.page_count):
            page=doc.load_page(page_num)
            pix=page.get_pixmap(dpi=300)
            page_path = os.path.join(pages_subdir, f'page_{page_num + 1}.jpeg')
            pix.save(page_path,'jpeg')
    print(f"Saved {doc.page_count} pages for: {filename}")

In [24]:
for filename in os.listdir(os.path.join(pages_dir, 'PORCONES.23.5 - 1628')):
    if filename.endswith('.jpeg'):
        print(f"Processing {filename}...")
        page_path = os.path.join(pages_dir, 'PORCONES.23.5 - 1628', filename)
        img=cv2.imread(page_path)
        if img.shape[1] > 13000:  # Arbitrary threshold to check if the page is a spread
            left_half=img[:, :img.shape[1]//2]
            right_half=img[:, img.shape[1]//2:]
            cv2.imwrite(os.path.join(pages_dir, 'PORCONES.23.5 - 1628', f"{os.path.splitext(filename)[0]}_left.jpeg"), left_half)
            cv2.imwrite(os.path.join(pages_dir, 'PORCONES.23.5 - 1628', f"{os.path.splitext(filename)[0]}_right.jpeg"), right_half)
            os.remove(page_path)

Processing page_1.jpeg...
Processing page_10.jpeg...
Processing page_11_left.jpeg...
Processing page_11_right.jpeg...
Processing page_12.jpeg...
Processing page_2.jpeg...
Processing page_3.jpeg...
Processing page_4.jpeg...
Processing page_5.jpeg...
Processing page_6.jpeg...
Processing page_7.jpeg...
Processing page_8.jpeg...
Processing page_9.jpeg...


In [ ]:
for filename in os.listdir(os.path.join(pages_dir, 'Buendia - Instruccion')):
    if filename.endswith('.jpeg'):
        print(f"Processing {filename}...")
        page_path = os.path.join(pages_dir, 'Buendia - Instruccion', filename)
        img=cv2.imread(page_path)
        print(img.shape)
        if img.shape[1] == 6955:  # Arbitrary threshold to check if the page is a spread
            left_half=img[:, :img.shape[1]//2]
            right_half=img[:, img.shape[1]//2:]
            cv2.imwrite(os.path.join(pages_dir, 'Buendia - Instruccion', f"{os.path.splitext(filename)[0]}_left.jpeg"), left_half)
            cv2.imwrite(os.path.join(pages_dir, 'Buendia - Instruccion', f"{os.path.splitext(filename)[0]}_right.jpeg"), right_half)
            os.remove(page_path)
        else:
            raise ValueError("")

In [32]:
book_to_i={}
for i,folder in enumerate(os.listdir(txt_dir)):
    if os.path.isdir(os.path.join(txt_dir,folder)):
        for filename in os.listdir(os.path.join(txt_dir,folder)):
            shutil.copy(os.path.join(txt_dir,folder,filename),os.path.join(r".\working data\transcripts", f"{i}_{filename}"))
    book_to_i[folder]=i
book_to_i

{'Buendia - Instruccion transcription': 0,
 'Covarrubias - Tesoro lengua transcription': 1,
 'Guardiola - Tratado nobleza transcription': 2,
 'PORCONES.228.38 - 1646 transcription': 3,
 'PORCONES.23.5 - 1628 transcription': 4,
 'PORCONES.748.6 – 1650 Transcription': 5}

In [ ]:
for folder in enumerate(os.listdir(txt_dir)):
    if os.path.isdir(os.path.join(txt_dir,folder)):
        for filename in os.listdir(os.path.join(txt_dir,folder)):
            shutil.copy(os.path.join(txt_dir,folder,filename),os.path.join(r".\working data\transcripts", f"{book_to_i[folder]}_{filename}"))
book_to_i





In [43]:
book_to_i['PORCONES.23.5 – 1628 transcription']=4
book_to_i['PORCONES.228.38 – 1646 transcription']=3

In [42]:
book_to_i

{'Buendia - Instruccion transcription': 0,
 'Covarrubias - Tesoro lengua transcription': 1,
 'Guardiola - Tratado nobleza transcription': 2,
 'PORCONES.228.38 - 1646 transcription': 3,
 'PORCONES.23.5 - 1628 transcription': 4,
 'PORCONES.748.6 – 1650 Transcription': 5,
 'PORCONES.23.5 – 1628 transcription': 4}

In [44]:
for folder in os.listdir(txt_dir):
    if os.path.isdir(os.path.join(txt_dir,folder)):
        for filename in os.listdir(os.path.join(txt_dir,folder)):
            pattern = re.compile(r"page_(\d+)(?:_(left|right))?\.txt")
            match = pattern.match(filename)
            page_num = match.group(1)
            n_folder=folder.replace(" transcription","")
            n_folder=n_folder.replace(" Transcription","")
            for img_filename in os.listdir(os.path.join(pages_dir, n_folder)):
                if img_filename.startswith(f"page_{page_num}.") or img_filename.startswith(f"page_{page_num}_"):
                    shutil.copy(os.path.join(pages_dir, n_folder, img_filename), os.path.join(r".\working data\images", f"{book_to_i[folder]}_{img_filename}"))

This gives us all the data in the desired form. Now, we would like to extract bounding boxes and have them labelled for model training.

In [45]:
!git clone https://github.com/clovaai/CRAFT-pytorch

Cloning into 'CRAFT-pytorch'...


We also need to download the model weights, I have done that manually.

In [54]:
!python CRAFT-pytorch/test.py --trained_model="CRAFT-pytorch\weights\craft_mlt_25k.pth" --test_folder=".\working data\images"

Loading weights from checkpoint (CRAFT-pytorch\weights\craft_mlt_25k.pth)
Test image 1/28: .\working data\images\0_page_2_left.jpeg
Test image 2/28: .\working data\images\0_page_2_right.jpeg
Test image 3/28: .\working data\images\0_page_3_left.jpeg
Test image 4/28: .\working data\images\0_page_3_right.jpeg
Test image 5/28: .\working data\images\0_page_4_left.jpeg
Test image 6/28: .\working data\images\0_page_4_right.jpeg
Test image 7/28: .\working data\images\1_page_7.jpeg
Test image 8/28: .\working data\images\1_page_8.jpeg
Test image 9/28: .\working data\images\1_page_9.jpeg
Test image 10/28: .\working data\images\2_page_12.jpeg
Test image 11/28: .\working data\images\2_page_13.jpeg
Test image 12/28: .\working data\images\2_page_14.jpeg
Test image 13/28: .\working data\images\3_page_1.jpeg
Test image 14/28: .\working data\images\3_page_2.jpeg
Test image 15/28: .\working data\images\3_page_3.jpeg
Test image 16/28: .\working data\images\3_page_4.jpeg
Test image 17/28: .\working data\im

c:\Users\reals\miniconda3\envs\ocr_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\reals\miniconda3\envs\ocr_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


In [13]:
img = cv2.imread(r'working data\images\0_page_2_right.jpeg')
print(img.shape)
values=open(r'bounding boxes\res_0_page_2_right.txt').readlines()
lines=[]
for line in values:
    if not line.strip():
        continue
    lines.append(list(map(int,line.strip().split(','))))
print(lines[1][:])

(5475, 3478, 3)
[1148, 379, 1947, 360, 1957, 750, 1158, 769]


I am not using Binarisation for the images because for some italic texts it produces bad results. Also, some ink bleeds corrupt the pages and I think grayscale will be better here. I am going to manually annotate these to ensure consistency in the dataset.

In [ ]:
boxes_dir="./bounding boxes"

def compare_rectangles(a,b):
    y1 = (a[1] + a[3] + a[5] + a[7]) / 4
    y2 = (b[1] + b[3] + b[5] + b[7]) / 4

    x1 = (a[0] + a[2] + a[4] + a[6]) / 4
    x2 = (b[0] + b[2] + b[4] + b[6]) / 4

    h1 = (
        math.hypot(a[0] - a[6], a[1] - a[7]) +
        math.hypot(a[2] - a[4], a[3] - a[5])
    ) / 2

    h2 = (
        math.hypot(b[0] - b[6], b[1] - b[7]) +
        math.hypot(b[2] - b[4], b[3] - b[5])
    ) / 2

    threshold = min(h1, h2) * 0.5

    if abs(y1 - y2) <= threshold:
        return x1 - x2

    return y1 - y2

def sort_boxes(rects):
    rects.sort(key=cmp_to_key(compare_rectangles))  
                            # y  x
def crop_image(img, rects):
    if img is None:
        print("Error: Image not found.")
        return None
    crops=[]
    if not rects:
        return rects
    for rect in rects:
        x1, y1, x2, y2 , x3, y3, x4, y4  = rect # Clockwise from top-left
        x_r=max(x2,x3)
        x_l=min(x1,x4)
        y_u=min(y1,y2)
        y_d=max(y3,y4)
        crop=img[y_u:y_d, x_l:x_r,:]
        crops.append(crop)
    return crops


prev_crops=[]
image_number=0
file_image_number={}

csv_file=open('final_dataset\labels.csv','a',newline='')
writer=csv.writer(csv_file)


for filename in os.listdir(boxes_dir):
    if filename.endswith('.txt'):
        
        print("Processing",filename)
        
        with open(os.path.join(boxes_dir,filename),'r') as f:
            lines=f.readlines()
            rects=[]
            for line in lines:
                if not line.strip():
                    continue
                rect=list(map(int, line.strip().split(','))) # It is really a csv under a .txt name
                rects.append(rect)
            if not rects:
                print("No boxes for file:",filename)
                continue
            
            sort_boxes(rects)
            
            img=cv2.imread(os.path.join(r".\working data\images", filename.replace('.txt','.jpeg').replace('res_','')))
            crops=crop_image(img, rects)
                        
            if not crops:
                raise ValueError("Image loading error:",filename)            
            
            
            transcript=open(os.path.join(r".\working data\transcripts", filename.replace('res_','')),'r').read().strip().split()
            i=0
            crop=0
            while crop<len(crops) and i<len(transcript):
                label=transcript[i]
                cv2.imwrite(os.path.join(r".\final_dataset\images", f"{image_number}_{i}.jpeg"), crops[crop])
                writer.writerow([f"{image_number}_{i}", label])
                i += 1
                crop+=1
            transcript.close()

            file_image_number[filename] = image_number
            image_number += 1
with open('file_image_number.json', 'w') as f:
    json.dump(file_image_number, f)

Processing res_0_page_2_left.txt
No boxes for file: res_0_page_2_left.txt
Processing res_0_page_2_right.txt
Label: Al
Index: 0
Surrounding text: ['Al', 'INFINITAMENTE', 'AMABLE', 'NIÃ‘O']
Enter the modulo for skip:


 1


Enter Y to proceed, N to enter label manually, C to skip this crop, L to skip this label:


 C


Label: Al
Index: 0
Surrounding text: ['Al', 'INFINITAMENTE', 'AMABLE', 'NIÃ‘O']
Enter the modulo for skip:


...

 18


Enter Y to proceed, N to enter label manually, C to skip this crop, L to skip this label:


 C


Label: de
Index: 18
Surrounding text: ['dignasteis', 'de', 'llamaros', 'Doctor', 'de', 'los', 'NiÃ±os,', 'sino']
Enter the modulo for skip:


KeyboardInterrupt: Interrupted by user